<a href="https://colab.research.google.com/github/AnDinh039/PST-PRIM---RMAT_128.edges---Julia/blob/main/PST-PRIM%20-%20RMAT_128_edges%20-%20JULIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# <img src="https://github.com/JuliaLang/julia-logo-graphics/raw/master/images/julia-logo-color.png" height="100" /> _Colab Notebook Template_

## Instructions
1. Work on a copy of this notebook: _File_ > _Save a copy in Drive_ (you will need a Google account). Alternatively, you can download the notebook using _File_ > _Download .ipynb_, then upload it to [Colab](https://colab.research.google.com/).
2. If you need a GPU: _Runtime_ > _Change runtime type_ > _Harware accelerator_ = _GPU_.
3. Execute the following cell (click on it and press Ctrl+Enter) to install Julia, IJulia and other packages (if needed, update `JULIA_VERSION` and the other parameters). This takes a couple of minutes.
4. Reload this page (press Ctrl+R, or ⌘+R, or the F5 key) and continue to the next section.

_Notes_:
* If your Colab Runtime gets reset (e.g., due to inactivity), repeat steps 2, 3 and 4.
* After installation, if you want to change the Julia version or activate/deactivate the GPU, you will need to reset the Runtime: _Runtime_ > _Factory reset runtime_ and repeat steps 3 and 4.

In [1]:
%%shell
set -e

#---------------------------------------------------#
JULIA_VERSION="1.8.2" # any version ≥ 0.7.0
JULIA_PACKAGES="IJulia BenchmarkTools"
JULIA_PACKAGES_IF_GPU="CUDA" # or CuArrays for older Julia versions
JULIA_NUM_THREADS=2
#---------------------------------------------------#

if [ -z `which julia` ]; then
  # Install Julia
  JULIA_VER=`cut -d '.' -f -2 <<< "$JULIA_VERSION"`
  echo "Installing Julia $JULIA_VERSION on the current Colab Runtime..."
  BASE_URL="https://julialang-s3.julialang.org/bin/linux/x64"
  URL="$BASE_URL/$JULIA_VER/julia-$JULIA_VERSION-linux-x86_64.tar.gz"
  wget -nv $URL -O /tmp/julia.tar.gz # -nv means "not verbose"
  tar -x -f /tmp/julia.tar.gz -C /usr/local --strip-components 1
  rm /tmp/julia.tar.gz

  # Install Packages
  nvidia-smi -L &> /dev/null && export GPU=1 || export GPU=0
  if [ $GPU -eq 1 ]; then
    JULIA_PACKAGES="$JULIA_PACKAGES $JULIA_PACKAGES_IF_GPU"
  fi
  for PKG in `echo $JULIA_PACKAGES`; do
    echo "Installing Julia package $PKG..."
    julia -e 'using Pkg; pkg"add '$PKG'; precompile;"' &> /dev/null
  done

  # Install kernel and rename it to "julia"
  echo "Installing IJulia kernel..."
  julia -e 'using IJulia; IJulia.installkernel("julia", env=Dict(
      "JULIA_NUM_THREADS"=>"'"$JULIA_NUM_THREADS"'"))'
  KERNEL_DIR=`julia -e "using IJulia; print(IJulia.kerneldir())"`
  KERNEL_NAME=`ls -d "$KERNEL_DIR"/julia*`
  mv -f $KERNEL_NAME "$KERNEL_DIR"/julia

  echo ''
  echo "Successfully installed `julia -v`!"
  echo "Please reload this page (press Ctrl+R, ⌘+R, or the F5 key) then"
  echo "jump to the 'Checking the Installation' section."
fi

Unrecognized magic `%%shell`.

Julia does not use the IPython `%magic` syntax.   To interact with the IJulia kernel, use `IJulia.somefunction(...)`, for example.  Julia macros, string macros, and functions can be used to accomplish most of the other functionalities of IPython magics.


# Checking the Installation
The `versioninfo()` function should print your Julia version and some other info about the system:

In [2]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 2 × AMD EPYC 7B12
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, znver2)
  GC: Built with stock GC
Threads: 2 default, 1 interactive, 2 GC (on 2 virtual cores)
Environment:
  JULIA_NUM_THREADS = auto


In [6]:
import Pkg

In [7]:
Pkg.add("BenchmarkTools")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed BenchmarkTools ─ v1.8.0
    Updating `~/.julia/environments/v1.12/Project.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [6e4b80f9] + BenchmarkTools v1.8.0
  [9abbd945] + Profile v1.11.0
Precompiling packages...
  13806.9 ms  ✓ BenchmarkTools
  1 dependency successfully precompiled in 23 seconds. 508 already precompiled.


In [8]:
using BenchmarkTools

M = rand(2^11, 2^11)

@btime $M * $M;

  404.411 ms (3 allocations: 32.00 MiB)


In [9]:
try
    using CUDA
catch
    println("No GPU found.")
else
    run(`nvidia-smi`)
    # Create a new random matrix directly on the GPU:
    M_on_gpu = CUDA.CURAND.rand(2^11, 2^11)
    @btime $M_on_gpu * $M_on_gpu; nothing
end

No GPU found.


In [12]:
using Printf
using Random

# ==============================================================================
# 0. HÀM ĐỌC FILE RMAT_128_edges.csv (VÀ TỰ SINH FILE NẾU CHƯA TỒN TẠI)
# ==============================================================================
function generate_rmat_128_csv(filepath::String; num_vertices::Int=128, num_edges::Int=1280)
    println("[Thông báo] Không tìm thấy 'filepath'.ĐangtạofilechuẩnRMAT-128(num_vertices đỉnh, $num_edges cạnh)...")
    Random.seed!(42)
    scale = Int(round(log2(num_vertices)))
    a, b, c, d = 0.57, 0.19, 0.19, 0.05
    edge_set = Dict{Tuple{Int, Int}, Float64}()

    # Đảm bảo đồ thị liên thông bằng cây khung cơ sở trước khi thêm cạnh RMAT
    for i in 1:(num_vertices - 1)
        edge_set[(i, i + 1)] = round(rand() * 99.0 + 1.0, digits=2)
    end

    while length(edge_set) < num_edges
        u, v = 0, 0
        for depth in (scale - 1):-1:0
            r = rand()
            bit_u = (r > a + b) ? 1 : 0
            bit_v = (r > a && r <= a + b) || (r > a + b + c) ? 1 : 0
            u |= (bit_u << depth)
            v |= (bit_v << depth)
        end
        u += 1; v += 1
        if u != v && !haskey(edge_set, (u, v)) && !haskey(edge_set, (v, u))
            edge_set[(u, v)] = round(rand() * 99.0 + 1.0, digits=2)
        end
    end

    open(filepath, "w") do io
        println(io, "src,dst,weight")
        for ((u, v), w) in sort(collect(edge_set))
            println(io, "u,v,$w")
        end
    end
    println("[Hoàn tất] Đã lưu '$filepath' thành công!\n")
end

function load_edges_csv(filepath::String)
    if !isfile(filepath)
        generate_rmat_128_csv(filepath)
    end

    edges = Tuple{Int, Int, Float64}[]
    open(filepath, "r") do io
        for line in eachline(io)
            s = strip(line)
            isempty(s) && continue
            # Hỗ trợ dấu phẩy, chấm phẩy, tab hoặc khoảng trắng
            parts = split(s, r"[,;\t\s]+")
            length(parts) < 2 && continue
            # Bỏ qua dòng header nếu cột đầu không phải là số nguyên
            u = tryparse(Int, parts[1])
            v = tryparse(Int, parts[2])
            (u === nothing || v === nothing) && continue
            # Nếu file chỉ có 2 cột (u, v) thì mặc định trọng số w = 1.0
            w = length(parts) >= 3 ? something(tryparse(Float64, parts[3]), 1.0) : 1.0
            push!(edges, (u, v, w))
        end
    end
    return edges
end


# ==============================================================================
# 1. TRƯỚC KHI TỐI ƯU (UNOPTIMIZED TEMPODL)
# - Giữ cột thời gian `t` bên trong mọi tuple: (t, x, y), (t, x), (t, w)
# - Giữ toàn bộ lịch sử các tầng strata {0}, {1}, ..., {t} trong Dict
# ==============================================================================
function prim_unoptimized(edges::Vector{Tuple{Int,Int,Float64}}; verbose::Bool=false, log_step::Int=25)
    uedge = Tuple{Int, Int, Float64}[]
    for (x, y, w) in edges
        push!(uedge, (x, y, w))
        push!(uedge, (y, x, w))
    end

    tree_edge_strata       = Dict{Int, Set{Tuple{Int, Int, Int}}}()     # {t} -> Set((t, x, y))
    tree_node_strata       = Dict{Int, Set{Tuple{Int, Int}}}()          # {t} -> Set((t, x))
    min_edge_weight_strata = Dict{Int, Set{Tuple{Int, Float64}}}()      # {t} -> Set((t, w))
    min_edge_strata        = Dict{Int, Set{Tuple{Int, Int, Int}}}()     # {t} -> Set((t, x, y))

    # Bước cơ sở (t = 0)
    min_w0 = minimum(e[3] for e in uedge)
    first_x, first_y = minimum((x, y) for (x, y, w) in uedge if w == min_w0)
    tree_edge_strata[0] = Set([(0, first_x, first_y)])

    t = 0
    while true
        # 1. tree_node{t}(t, x) += tree_edge{t}(t, x, y) + tree_edge{t}(t, y, x)
        nodes_t = Set{Tuple{Int, Int}}()
        for (step_t, x, y) in tree_edge_strata[t]
            push!(nodes_t, (step_t, x))
            push!(nodes_t, (step_t, y))
        end
        tree_node_strata[t] = nodes_t

        # 2. min_edge_weight{t}(t, min(w))
        cand_weights = Float64[]
        for (x, y, w) in uedge
            if ((t, x) in tree_node_strata[t]) && !((t, y) in tree_node_strata[t])
                push!(cand_weights, w)
            end
        end

        if isempty(cand_weights)
            if verbose
                total_tuples = sum(length(v) for v in values(tree_edge_strata)) +
                               sum(length(v) for v in values(tree_node_strata)) +
                               sum(length(v) for v in values(min_edge_strata))
                println("  [Kết thúc ở t=$t] Đã phủ kín cây khung! Tổng strata: $(length(tree_edge_strata)) | Tổng tuples trong RAM: $total_tuples")
            end
            break
        end

        min_w = minimum(cand_weights)
        min_edge_weight_strata[t] = Set([(t, min_w)])

        # 3. min_edge{t}(t, x, y)
        chosen_candidates = Tuple{Int, Int}[]
        for (_, w_val) in min_edge_weight_strata[t]
            for (x, y, w) in uedge
                if w == w_val && ((t, x) in tree_node_strata[t]) && !((t, y) in tree_node_strata[t])
                    push!(chosen_candidates, (x, y))
                end
            end
        end
        best_x, best_y = minimum(chosen_candidates)
        min_edge_strata[t] = Set([(t, best_x, best_y)])

        # 4. tree_edge{t+1}(t+1, x, y) += tree_edge{t}(t, x, y) + min_edge{t}(t, x, y)
        next_edges = Set{Tuple{Int, Int, Int}}()
        for (_, x, y) in tree_edge_strata[t]
            push!(next_edges, (t + 1, x, y))
        end
        for (_, x, y) in min_edge_strata[t]
            push!(next_edges, (t + 1, x, y))
        end
        tree_edge_strata[t + 1] = next_edges

        if verbose && (t < 3 || t % log_step == 0)
            total_tuples = sum(length(v) for v in values(tree_edge_strata)) +
                           sum(length(v) for v in values(tree_node_strata)) +
                           sum(length(v) for v in values(min_edge_strata))
            @printf("  [Bước t=%3d] +Cạnh (%3d, %3d) w=%6.2f | Strata đang lưu: %3d | Tổng tuples trong RAM: %6d\n",
                    t, best_x, best_y, min_w, length(tree_edge_strata), total_tuples)
        end

        t += 1
    end

    result = [(x, y) for (_, x, y) in tree_edge_strata[t]]
    ram_bytes = Base.summarysize(tree_edge_strata) + Base.summarysize(tree_node_strata) +
                Base.summarysize(min_edge_weight_strata) + Base.summarysize(min_edge_strata)
    return result, ram_bytes, t
end


# ==============================================================================
# 2. SAU KHI TỐI ƯU SUBSUMPTION (Mục 5.1)
# - Đã XÓA cột thời gian `t` bên trong các tuple: còn (x, y), (x), (w)
# - VẪN GIỮ toàn bộ các tầng strata {0}, {1}, ..., {t} trong bộ nhớ
# ==============================================================================
function prim_subsumption(edges::Vector{Tuple{Int,Int,Float64}}; verbose::Bool=false, log_step::Int=25)
    uedge = Tuple{Int, Int, Float64}[]
    for (x, y, w) in edges
        push!(uedge, (x, y, w))
        push!(uedge, (y, x, w))
    end

    tree_edge_strata       = Dict{Int, Set{Tuple{Int, Int}}}() # {t} -> Set((x, y))
    tree_node_strata       = Dict{Int, Set{Int}}()             # {t} -> Set(x)
    min_edge_weight_strata = Dict{Int, Float64}()              # {t} -> w
    min_edge_strata        = Dict{Int, Tuple{Int, Int}}()      # {t} -> (x, y)

    min_w0 = minimum(e[3] for e in uedge)
    first_edge = minimum((x, y) for (x, y, w) in uedge if w == min_w0)
    tree_edge_strata[0] = Set([first_edge])

    t = 0
    while true
        # 1. tree_node{t}(x) := tree_edge{t}(x, y) + tree_edge{t}(y, x)
        nodes_t = Set{Int}()
        for (x, y) in tree_edge_strata[t]
            push!(nodes_t, x)
            push!(nodes_t, y)
        end
        tree_node_strata[t] = nodes_t

        # 2. min_edge_weight{t}(min(w))
        cand_weights = Float64[]
        for (x, y, w) in uedge
            if (x in tree_node_strata[t]) && !(y in tree_node_strata[t])
                push!(cand_weights, w)
            end
        end

        if isempty(cand_weights)
            if verbose
                total_tuples = sum(length(v) for v in values(tree_edge_strata)) +
                               sum(length(v) for v in values(tree_node_strata)) +
                               length(min_edge_strata)
                println("  [Kết thúc ở t=$t] Đã phủ kín cây khung! Tổng strata: $(length(tree_edge_strata)) | Tổng tuples trong RAM: $total_tuples")
            end
            break
        end

        min_w = minimum(cand_weights)
        min_edge_weight_strata[t] = min_w

        # 3. min_edge{t}(x, y)
        best_edge = minimum((x, y) for (x, y, w) in uedge
                            if w == min_w && (x in tree_node_strata[t]) && !(y in tree_node_strata[t]))
        min_edge_strata[t] = best_edge

        # 4. tree_edge{t+1}(x, y) := tree_edge{t}(x, y) + min_edge{t}(x, y)
        next_edges = copy(tree_edge_strata[t])
        push!(next_edges, min_edge_strata[t])
        tree_edge_strata[t + 1] = next_edges

        if verbose && (t < 3 || t % log_step == 0)
            total_tuples = sum(length(v) for v in values(tree_edge_strata)) +
                           sum(length(v) for v in values(tree_node_strata)) +
                           length(min_edge_strata)
            @printf("  [Bước t=%3d] +Cạnh (%3d, %3d) w=%6.2f | Strata đang lưu: %3d | Tổng tuples trong RAM: %6d\n",
                    t, best_edge[1], best_edge[2], min_w, length(tree_edge_strata), total_tuples)
        end

        t += 1
    end

    result = collect(tree_edge_strata[t])
    ram_bytes = Base.summarysize(tree_edge_strata) + Base.summarysize(tree_node_strata) +
                Base.summarysize(min_edge_weight_strata) + Base.summarysize(min_edge_strata)
    return result, ram_bytes, t
end


# ==============================================================================
# 3. SAU KHI TỐI ƯU SUBSUMPTION + TEMPORAL ELIMINATION (Mục 5.2)
# - Xóa bỏ hoàn toàn các tầng lịch sử {0}, {1}, ..., {t}
# - Chỉ giữ trạng thái hiện tại `tree_edge` và `tree_edge_prime`
# ==============================================================================
function prim_temporal_elimination(edges::Vector{Tuple{Int,Int,Float64}}; verbose::Bool=false, log_step::Int=25)
    uedge = Tuple{Int, Int, Float64}[]
    for (x, y, w) in edges
        push!(uedge, (x, y, w))
        push!(uedge, (y, x, w))
    end

    min_w0 = minimum(e[3] for e in uedge)
    first_edge = minimum((x, y) for (x, y, w) in uedge if w == min_w0)

    tree_edge_prime = Set{Tuple{Int, Int}}([first_edge])
    tree_edge       = Set{Tuple{Int, Int}}()
    tree_node       = Set{Int}()

    t = 0
    while true
        # Ghi đè trạng thái bước trước: tree_edge := tree_edge'
        tree_edge = copy(tree_edge_prime)

        empty!(tree_node)
        for (x, y) in tree_edge
            push!(tree_node, x)
            push!(tree_node, y)
        end

        best_w = Inf
        best_edge = (0, 0)
        found = false

        for (x, y, w) in uedge
            if (x in tree_node) && !(y in tree_node)
                if !found || (w, x, y) < (best_w, best_edge[1], best_edge[2])
                    best_w = w
                    best_edge = (x, y)
                    found = true
                end
            end
        end

        if !found
            if verbose
                total_tuples = length(tree_edge_prime) + length(tree_node)
                println("  [Kết thúc ở t=$t] Đã phủ kín cây khung! Strata lưu lịch sử: 0 | Tổng tuples trong RAM: $total_tuples")
            end
            break
        end

        tree_edge_prime = copy(tree_edge)
        push!(tree_edge_prime, best_edge)

        if verbose && (t < 3 || t % log_step == 0)
            total_tuples = length(tree_edge_prime) + length(tree_node) + 1
            @printf("  [Bước t=%3d] +Cạnh (%3d, %3d) w=%6.2f | Strata đang lưu:   1 | Tổng tuples trong RAM: %6d\n",
                    t, best_edge[1], best_edge[2], best_w, total_tuples)
        end

        t += 1
    end

    result = collect(tree_edge_prime)
    ram_bytes = Base.summarysize(tree_edge_prime) + Base.summarysize(tree_edge) + Base.summarysize(tree_node)
    return result, ram_bytes, t
end


# ==============================================================================
# CHƯƠNG TRÌNH CHÍNH: ĐỌC RMAT_128_edges.csv, IN TỪNG BƯỚC & ĐO HIỆU NĂNG
# ==============================================================================
const CSV_FILE = "RMAT_128_edges.csv"
rmat_edges = load_edges_csv(CSV_FILE)

nodes_set = Set{Int}()
for (u, v, _) in rmat_edges
    push!(nodes_set, u); push!(nodes_set, v)
end

println("==========================================================================================")
println("THÔNG TIN DATASET: $CSV_FILE")
println("  - Số đỉnh (Vertices): $(length(nodes_set))")
println("  - Số cạnh có hướng (Edges): $(length(rmat_edges)) (Vô hướng uedge: $(2 * length(rmat_edges)))")
println("==========================================================================================")

println("\n--- [1] DIỄN GIẢI TỪNG BƯỚC: UNOPTIMIZED TEMPODL ---")
prim_unoptimized(rmat_edges, verbose=true, log_step=25)

println("\n--- [2] DIỄN GIẢI TỪNG BƯỚC: SAU TỐI ƯU SUBSUMPTION ---")
prim_subsumption(rmat_edges, verbose=true, log_step=25)

println("\n--- [3] DIỄN GIẢI TỪNG BƯỚC: SAU TỐI ƯU SUBSUMPTION + TEMPORAL ELIMINATION ---")
prim_temporal_elimination(rmat_edges, verbose=true, log_step=25)

# Warm-up JIT Compiler của Julia trước khi bấm giờ
prim_unoptimized(rmat_edges[1:min(20, end)])
prim_subsumption(rmat_edges[1:min(20, end)])
prim_temporal_elimination(rmat_edges[1:min(20, end)])

# Đo lường thực tế trên toàn bộ RMAT_128_edges.csv (không bật verbose để đo thời gian chuẩn)
stats1 = @timed prim_unoptimized(rmat_edges, verbose=false)
stats2 = @timed prim_subsumption(rmat_edges, verbose=false)
stats3 = @timed prim_temporal_elimination(rmat_edges, verbose=false)

mst1, ram1, steps1 = stats1.value
mst2, ram2, steps2 = stats2.value
mst3, ram3, steps3 = stats3.value

println("\n==========================================================================================")
println("BẢNG TỔNG KẾT TRÊN BỘ DỮ LIỆU RMAT_128_edges.csv (Số bước lặp: $steps3, Số cạnh MST: $(length(mst3)))")
println("==========================================================================================")
@printf("%-38s | %14s | %17s | %12s\n", "Phiên bản", "RAM lưu IDBs", "Tổng cấp phát Heap", "Thời gian")
println("-"^90)
@printf("%-38s | %11.2f KB | %14.2f MB | %9.2f ms\n",
        "1. Unoptimized TempoDL", ram1/1024, stats1.bytes/(1024^2), stats1.time*1000)
@printf("%-38s | %11.2f KB | %14.2f MB | %9.2f ms\n",
        "2. Subsumption Only", ram2/1024, stats2.bytes/(1024^2), stats2.time*1000)
@printf("%-38s | %11.2f KB | %14.2f MB | %9.2f ms\n",
        "3. Subsumption + Temporal Elimination", ram3/1024, stats3.bytes/(1024^2), stats3.time*1000)
println("==========================================================================================")
println("Kiểm tra tính đúng đắn (3 kết quả MST giống hệt nhau): ", Set(mst1) == Set(mst2) == Set(mst3))





THÔNG TIN DATASET: RMAT_128_edges.csv
  - Số đỉnh (Vertices): 128
  - Số cạnh có hướng (Edges): 1280 (Vô hướng uedge: 2560)

--- [1] DIỄN GIẢI TỪNG BƯỚC: UNOPTIMIZED TEMPODL ---
  [Bước t=  0] +Cạnh ( 84,  22) w=  7.00 | Strata đang lưu:   2 | Tổng tuples trong RAM:      6
  [Bước t=  1] +Cạnh ( 22,  45) w=  6.00 | Strata đang lưu:   3 | Tổng tuples trong RAM:     13
  [Bước t=  2] +Cạnh ( 22,  93) w=  9.00 | Strata đang lưu:   4 | Tổng tuples trong RAM:     22
  [Bước t= 25] +Cạnh ( 38,  39) w= 11.00 | Strata đang lưu:  27 | Tổng tuples trong RAM:    781
  [Bước t= 50] +Cạnh ( 49,  66) w=  6.00 | Strata đang lưu:  52 | Tổng tuples trong RAM:   2806
  [Bước t= 75] +Cạnh (104,  30) w=  3.00 | Strata đang lưu:  77 | Tổng tuples trong RAM:   6081
  [Bước t=100] +Cạnh ( 73, 117) w= 10.00 | Strata đang lưu: 102 | Tổng tuples trong RAM:  10606
  [Bước t=125] +Cạnh ( 22,  23) w= 33.00 | Strata đang lưu: 127 | Tổng tuples trong RAM:  16381
  [Kết thúc ở t=126] Đã phủ kín cây khung! Tổng strata